In [8]:
import pandas as pd

df = pd.read_csv('/projects/retprogression/rgarridogarcia/ensemble/ensemble_all_methods/comprehensive_summary_table.csv')

display(df)

,model,type,balanced_accuracy,sensitivity,specificity,f1_score,auc,TP,FP,FN,TN
0,Model 1,Individual,0.9511 / 0.7899 / NaN,0.9719 / 0.9901 / NaN,0.9302 / 0.5897 / NaN,0.9850 / 0.9901 / NaN,0.9860 / 0.8745 / NaN,- / 1596 / -,- / 16 / -,- / 16 / -,- / 23 / -
1,Model 2,Individual,0.8827 / 0.8438 / NaN,0.9803 / 0.9696 / NaN,0.7850 / 0.7179 / NaN,0.9872 / 0.9812 / NaN,0.9801 / 0.8872 / NaN,- / 1563 / -,- / 11 / -,- / 49 / -,- / 28 / -
2,Model 3,Individual,0.9450 / 0.8337 / NaN,0.9734 / 0.9752 / NaN,0.9167 / 0.6923 / NaN,0.9854 / 0.9837 / NaN,0.9848 / 0.8994 / NaN,3847 / 1572 / -,9 / 12 / -,105 / 40 / -,99 / 27 / -
3,Model 4,Individual,0.9062 / 0.8087 / NaN,0.9717 / 0.9764 / NaN,0.8407 / 0.6410 / NaN,0.9835 / 0.9837 / NaN,0.9821 / 0.8925 / NaN,- / 1574 / -,- / 14 / -,- / 38 / -,- / 25 / -
4,Model 5,Individual,0.9333 / 0.8487 / NaN,0.9754 / 0.9795 / NaN,0.8911 / 0.7179 / NaN,0.9862 / 0.9863 / NaN,0.9889 / 0.8948 / NaN,3813 / 1579 / -,11 / 11 / -,96 / 33 / -,90 / 28 / -
5,Ensemble (average),Ensemble,NaN / 0.8497 / 0.8554,NaN / 0.9814 / 0.9671,NaN / 0.7179 / 0.7436,NaN / 0.9872 / 0.9802,NaN / 0.8987 / 0.8987,- / 1582 / 1559,- / 11 / 10,- / 30 / 53,- / 28 / 29
6,Ensemble (confidence),Ensemble,NaN / 0.8494 / 0.8550,NaN / 0.9808 / 0.9665,NaN / 0.7179 / 0.7436,NaN / 0.9869 / 0.9799,NaN / 0.8986 / 0.8986,- / 1581 / 1558,- / 11 / 10,- / 31 / 54,- / 28 / 29
7,Ensemble (logit),Ensemble,NaN / 0.8497 / 0.8550,NaN / 0.9814 / 0.9665,NaN / 0.7179 / 0.7436,NaN / 0.9872 / 0.9799,NaN / 0.8975 / 0.8975,- / 1582 / 1558,- / 11 / 10,- / 30 / 54,- / 28 / 29
8,Ensemble (majority_60),Ensemble,NaN / 0.8497 / 0.8554,NaN / 0.9814 / 0.9671,NaN / 0.7179 / 0.7436,NaN / 0.9872 / 0.9802,NaN / 0.8987 / 0.8987,- / 1582 / 1559,- / 11 / 10,- / 30 / 53,- / 28 / 29
9,Ensemble (majority_80),Ensemble,NaN / 0.8497 / 0.8554,NaN / 0.9814 / 0.9671,NaN / 0.7179 / 0.7436,NaN / 0.9872 / 0.9802,NaN / 0.8987 / 0.8987,- / 1582 / 1559,- / 11 / 10,- / 30 / 53,- / 28 / 29


In [10]:
import pandas as pd
import numpy as np


def parse_triple_value(value_str):
    """Parse 'val1 / val2 / val3' format into list of floats"""
    if pd.isna(value_str):
        return [np.nan, np.nan, np.nan]
    
    parts = str(value_str).split(' / ')
    result = []
    for part in parts:
        part = part.strip()
        if part == 'NaN' or part == '-' or part == '':
            result.append(np.nan)
        else:
            try:
                result.append(float(part))
            except:
                result.append(np.nan)
    
    # Ensure we have 3 values
    while len(result) < 3:
        result.append(np.nan)
    
    return result[:3]

def format_triple_value(values):
    """Format list of 3 values back to string"""
    formatted = []
    for v in values:
        if np.isnan(v) or v is None:
            formatted.append('-')
        else:
            formatted.append(f"{v:.4f}" if isinstance(v, float) and v < 1 else str(int(v)))
    return ' / '.join(formatted)

# Parse all metrics
for col in ['sensitivity', 'specificity', 'balanced_accuracy', 'f1_score', 'auc', 'TP', 'FP', 'FN', 'TN']:
    if col in df.columns:
        df[f'{col}_parsed'] = df[col].apply(parse_triple_value)

# Create class 0 metrics
results_class0 = []

for idx, row in df.iterrows():
    # Get parsed values
    sens = row['sensitivity_parsed']
    spec = row['specificity_parsed']
    bal_acc = row['balanced_accuracy_parsed']
    auc = row['auc_parsed']
    tp = row['TP_parsed']
    fp = row['FP_parsed']
    fn = row['FN_parsed']
    tn = row['TN_parsed']
    
    # Swap for class 0
    sens_class0 = spec  # Sensitivity for class 0 = Specificity for class 1
    spec_class0 = sens  # Specificity for class 0 = Sensitivity for class 1
    bal_acc_class0 = bal_acc  # Stays the same
    
    # Swap confusion matrix
    tp_class0 = tn
    fp_class0 = fn
    fn_class0 = fp
    tn_class0 = tp
    
    # Calculate precision and F1 for class 0 (for each test set)
    prec_class0 = []
    f1_class0 = []
    auc_class0 = []
    
    for i in range(3):
        # Precision for class 0
        if tp_class0[i] + fp_class0[i] > 0:
            prec = tp_class0[i] / (tp_class0[i] + fp_class0[i])
        else:
            prec = np.nan
        prec_class0.append(prec)
        
        # F1 for class 0
        if prec + sens_class0[i] > 0:
            f1 = 2 * (prec * sens_class0[i]) / (prec + sens_class0[i])
        else:
            f1 = np.nan
        f1_class0.append(f1)
        
        # AUC for class 0 = 1 - AUC for class 1
        if not np.isnan(auc[i]):
            auc_class0.append(1 - auc[i])
        else:
            auc_class0.append(np.nan)
    
    # Store results
    results_class0.append({
        'model': row['model'],
        'type': row['type'],
        'balanced_accuracy': format_triple_value(bal_acc_class0),
        'sensitivity': format_triple_value(sens_class0),
        'specificity': format_triple_value(spec_class0),
        'precision': format_triple_value(prec_class0),
        'f1_score': format_triple_value(f1_class0),
        'auc': format_triple_value(auc_class0),
        'TP': format_triple_value(tp_class0),
        'FP': format_triple_value(fp_class0),
        'FN': format_triple_value(fn_class0),
        'TN': format_triple_value(tn_class0)
    })

# Create DataFrame for class 0
df_class0 = pd.DataFrame(results_class0)

# Save
df_class0.to_csv('ensemble_comparison_class0.csv', index=False)

print("Class 0 metrics saved to 'ensemble_comparison_class0.csv'")
print("\nPreview:")
display(df_class0)

Class 0 metrics saved to 'ensemble_comparison_class0.csv'

Preview:


,model,type,balanced_accuracy,sensitivity,specificity,precision,f1_score,auc,TP,FP,FN,TN
0,Model 1,Individual,0.9511 / 0.7899 / -,0.9302 / 0.5897 / -,0.9719 / 0.9901 / -,- / 0.5897 / -,- / 0.5897 / -,0.0140 / 0.1255 / -,- / 23 / -,- / 16 / -,- / 16 / -,- / 1596 / -
1,Model 2,Individual,0.8827 / 0.8438 / -,0.7850 / 0.7179 / -,0.9803 / 0.9696 / -,- / 0.3636 / -,- / 0.4827 / -,0.0199 / 0.1128 / -,- / 28 / -,- / 49 / -,- / 11 / -,- / 1563 / -
2,Model 3,Individual,0.9450 / 0.8337 / -,0.9167 / 0.6923 / -,0.9734 / 0.9752 / -,0.4853 / 0.4030 / -,0.6346 / 0.5094 / -,0.0152 / 0.1006 / -,99 / 27 / -,105 / 40 / -,9 / 12 / -,3847 / 1572 / -
3,Model 4,Individual,0.9062 / 0.8087 / -,0.8407 / 0.6410 / -,0.9717 / 0.9764 / -,- / 0.3968 / -,- / 0.4902 / -,0.0179 / 0.1075 / -,- / 25 / -,- / 38 / -,- / 14 / -,- / 1574 / -
4,Model 5,Individual,0.9333 / 0.8487 / -,0.8911 / 0.7179 / -,0.9754 / 0.9795 / -,0.4839 / 0.4590 / -,0.6272 / 0.5600 / -,0.0111 / 0.1052 / -,90 / 28 / -,96 / 33 / -,11 / 11 / -,3813 / 1579 / -
5,Ensemble (average),Ensemble,- / 0.8497 / 0.8554,- / 0.7179 / 0.7436,- / 0.9814 / 0.9671,- / 0.4828 / 0.3537,- / 0.5773 / 0.4793,- / 0.1013 / 0.1013,- / 28 / 29,- / 30 / 53,- / 11 / 10,- / 1582 / 1559
6,Ensemble (confidence),Ensemble,- / 0.8494 / 0.8550,- / 0.7179 / 0.7436,- / 0.9808 / 0.9665,- / 0.4746 / 0.3494,- / 0.5714 / 0.4754,- / 0.1014 / 0.1014,- / 28 / 29,- / 31 / 54,- / 11 / 10,- / 1581 / 1558
7,Ensemble (logit),Ensemble,- / 0.8497 / 0.8550,- / 0.7179 / 0.7436,- / 0.9814 / 0.9665,- / 0.4828 / 0.3494,- / 0.5773 / 0.4754,- / 0.1025 / 0.1025,- / 28 / 29,- / 30 / 54,- / 11 / 10,- / 1582 / 1558
8,Ensemble (majority_60),Ensemble,- / 0.8497 / 0.8554,- / 0.7179 / 0.7436,- / 0.9814 / 0.9671,- / 0.4828 / 0.3537,- / 0.5773 / 0.4793,- / 0.1013 / 0.1013,- / 28 / 29,- / 30 / 53,- / 11 / 10,- / 1582 / 1559
9,Ensemble (majority_80),Ensemble,- / 0.8497 / 0.8554,- / 0.7179 / 0.7436,- / 0.9814 / 0.9671,- / 0.4828 / 0.3537,- / 0.5773 / 0.4793,- / 0.1013 / 0.1013,- / 28 / 29,- / 30 / 53,- / 11 / 10,- / 1582 / 1559


In [15]:
import pandas as pd
import numpy as np

# Load your class 0 results
df = pd.read_csv('ensemble_comparison_class0.csv')

def parse_triple_value(value_str):
    """Parse 'val1 / val2 / val3' format"""
    if pd.isna(value_str):
        return [np.nan, np.nan, np.nan]
    
    parts = str(value_str).split('/')
    result = []
    for part in parts:
        part = part.strip()
        if part == '-' or part == '' or part == 'NaN':
            result.append(np.nan)
        else:
            try:
                result.append(float(part))
            except:
                result.append(np.nan)
    
    while len(result) < 3:
        result.append(np.nan)
    
    return result[:3]

def format_triple_value(values, is_metric=False):
    """Format list back to string"""
    formatted = []
    for v in values:
        if np.isnan(v) or v is None:
            formatted.append('-')
        else:
            if is_metric:
                formatted.append(f"{v:.4f}")
            else:
                formatted.append(str(int(round(v))))
    return ' / '.join(formatted)

# Parse all metrics
for col in ['sensitivity', 'specificity', 'precision', 'f1_score', 'auc', 'TP', 'FP', 'FN', 'TN']:
    if col in df.columns:
        df[f'{col}_parsed'] = df[col].apply(parse_triple_value)

# Infer total P and N from Model 3
model3_idx = df[df['model'] == 'Model 3'].index[0]
tp3 = df.loc[model3_idx, 'TP_parsed']
fn3 = df.loc[model3_idx, 'FN_parsed']
tn3 = df.loc[model3_idx, 'TN_parsed']
fp3 = df.loc[model3_idx, 'FP_parsed']

total_P = []
total_N = []
for i in range(3):
    if not np.isnan(tp3[i]) and not np.isnan(fn3[i]):
        total_P.append(int(tp3[i] + fn3[i]))
    else:
        total_P.append(np.nan)
    
    if not np.isnan(tn3[i]) and not np.isnan(fp3[i]):
        total_N.append(int(tn3[i] + fp3[i]))
    else:
        total_N.append(np.nan)

print("Inferred totals from Model 3:")
print(f"Total Positives per test set: {total_P}")
print(f"Total Negatives per test set: {total_N}")
print()

results_fixed = []

for idx, row in df.iterrows():
    sens = row['sensitivity_parsed']
    spec = row['specificity_parsed']
    
    # Use EXISTING confusion matrix values when available
    tp_orig = row['TP_parsed']
    fp_orig = row['FP_parsed']
    fn_orig = row['FN_parsed']
    tn_orig = row['TN_parsed']
    
    # Arrays for all metrics
    tp_final = []
    fp_final = []
    fn_final = []
    tn_final = []
    precision_final = []
    f1_final = []
    auc_final = []
    
    for i in range(3):
        # Use existing values if available, otherwise calculate
        if not np.isnan(tp_orig[i]):
            tp = tp_orig[i]
            fn = fn_orig[i]
            tn = tn_orig[i]
            fp = fp_orig[i]
        else:
            # Calculate from sensitivity/specificity
            if not np.isnan(sens[i]) and not np.isnan(total_P[i]):
                tp = round(sens[i] * total_P[i])
                fn = total_P[i] - tp
            else:
                tp = np.nan
                fn = np.nan
            
            if not np.isnan(spec[i]) and not np.isnan(total_N[i]):
                tn = round(spec[i] * total_N[i])
                fp = total_N[i] - tn
            else:
                tn = np.nan
                fp = np.nan
        
        tp_final.append(tp)
        fp_final.append(fp)
        fn_final.append(fn)
        tn_final.append(tn)
        
        # Calculate precision
        if not np.isnan(tp) and not np.isnan(fp) and (tp + fp) > 0:
            precision = tp / (tp + fp)
        else:
            precision = np.nan
        precision_final.append(precision)
        
        # Calculate F1 score
        if not np.isnan(precision) and not np.isnan(sens[i]) and (precision + sens[i]) > 0:
            f1 = 2 * (precision * sens[i]) / (precision + sens[i])
        else:
            f1 = np.nan
        f1_final.append(f1)
        
        # Calculate AUC for class 0
        # Note: You need the original AUC from class 1 predictions
        # AUC for class 0 = 1 - AUC for class 1
        # If you don't have the original, this will remain as calculated before
        auc_orig = row.get('auc_parsed', [np.nan, np.nan, np.nan])
        if i < len(auc_orig) and not np.isnan(auc_orig[i]):
            auc_final.append(auc_orig[i])
        else:
            auc_final.append(np.nan)
    
    # Create result row
    result_row = {
        'model': row['model'],
        'type': row['type'],
        'balanced_accuracy': row['balanced_accuracy'],
        'sensitivity': row['sensitivity'],
        'specificity': row['specificity'],
        'precision': format_triple_value(precision_final, is_metric=True),
        'f1_score': format_triple_value(f1_final, is_metric=True),
        'auc': format_triple_value(auc_final, is_metric=True),
        'TP': format_triple_value(tp_final),
        'FP': format_triple_value(fp_final),
        'FN': format_triple_value(fn_final),
        'TN': format_triple_value(tn_final)
    }
    
    results_fixed.append(result_row)

# Create fixed DataFrame
df_fixed = pd.DataFrame(results_fixed)

# Save
df_fixed.to_csv('ensemble_comparison_class0_complete.csv', index=False)

print("Complete results saved to 'ensemble_comparison_class0_complete.csv'")
print("\nPreview:")
display(df_fixed)

Inferred totals from Model 3:
Total Positives per test set: [108, 39, nan]
Total Negatives per test set: [3952, 1612, nan]

Complete results saved to 'ensemble_comparison_class0_complete.csv'

Preview:


,model,type,balanced_accuracy,sensitivity,specificity,precision,f1_score,auc,TP,FP,FN,TN
0,Model 1,Individual,0.9511 / 0.7899 / -,0.9302 / 0.5897 / -,0.9719 / 0.9901 / -,0.4739 / 0.5897 / -,0.6279 / 0.5897 / -,0.0140 / 0.1255 / -,100 / 23 / -,111 / 16 / -,8 / 16 / -,3841 / 1596 / -
1,Model 2,Individual,0.8827 / 0.8438 / -,0.7850 / 0.7179 / -,0.9803 / 0.9696 / -,0.5215 / 0.3636 / -,0.6267 / 0.4827 / -,0.0199 / 0.1128 / -,85 / 28 / -,78 / 49 / -,23 / 11 / -,3874 / 1563 / -
2,Model 3,Individual,0.9450 / 0.8337 / -,0.9167 / 0.6923 / -,0.9734 / 0.9752 / -,0.4853 / 0.4030 / -,0.6346 / 0.5094 / -,0.0152 / 0.1006 / -,99 / 27 / -,105 / 40 / -,9 / 12 / -,3847 / 1572 / -
3,Model 4,Individual,0.9062 / 0.8087 / -,0.8407 / 0.6410 / -,0.9717 / 0.9764 / -,0.4483 / 0.3968 / -,0.5848 / 0.4902 / -,0.0179 / 0.1075 / -,91 / 25 / -,112 / 38 / -,17 / 14 / -,3840 / 1574 / -
4,Model 5,Individual,0.9333 / 0.8487 / -,0.8911 / 0.7179 / -,0.9754 / 0.9795 / -,0.4839 / 0.4590 / -,0.6272 / 0.5600 / -,0.0111 / 0.1052 / -,90 / 28 / -,96 / 33 / -,11 / 11 / -,3813 / 1579 / -
5,Ensemble (average),Ensemble,- / 0.8497 / 0.8554,- / 0.7179 / 0.7436,- / 0.9814 / 0.9671,- / 0.4828 / 0.3537,- / 0.5773 / 0.4793,- / 0.1013 / 0.1013,- / 28 / 29,- / 30 / 53,- / 11 / 10,- / 1582 / 1559
6,Ensemble (confidence),Ensemble,- / 0.8494 / 0.8550,- / 0.7179 / 0.7436,- / 0.9808 / 0.9665,- / 0.4746 / 0.3494,- / 0.5714 / 0.4754,- / 0.1014 / 0.1014,- / 28 / 29,- / 31 / 54,- / 11 / 10,- / 1581 / 1558
7,Ensemble (logit),Ensemble,- / 0.8497 / 0.8550,- / 0.7179 / 0.7436,- / 0.9814 / 0.9665,- / 0.4828 / 0.3494,- / 0.5773 / 0.4754,- / 0.1025 / 0.1025,- / 28 / 29,- / 30 / 54,- / 11 / 10,- / 1582 / 1558
8,Ensemble (majority_60),Ensemble,- / 0.8497 / 0.8554,- / 0.7179 / 0.7436,- / 0.9814 / 0.9671,- / 0.4828 / 0.3537,- / 0.5773 / 0.4793,- / 0.1013 / 0.1013,- / 28 / 29,- / 30 / 53,- / 11 / 10,- / 1582 / 1559
9,Ensemble (majority_80),Ensemble,- / 0.8497 / 0.8554,- / 0.7179 / 0.7436,- / 0.9814 / 0.9671,- / 0.4828 / 0.3537,- / 0.5773 / 0.4793,- / 0.1013 / 0.1013,- / 28 / 29,- / 30 / 53,- / 11 / 10,- / 1582 / 1559
